# 1. Create the silver schema and managed location in Unity Catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuresalesdatabricks.silver
MANAGED LOCATION 'abfss://silver@azuresales.dfs.core.windows.net/';

# 2. Create the silver table with an explicit schema for sales_orders

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.sales_orders (
  order_id        STRING,
  order_date      DATE,
  customer_id     STRING,
  store_id        STRING,
  rep_id          STRING,
  order_status    STRING,
  payment_method  STRING,
  order_total     DOUBLE,
  _source_file    STRING,
  _ingested_at    TIMESTAMP
) USING DELTA;

# 3. Create the silver table with an explicit schema for order_items

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.order_items (
  order_item_id STRING, 
  order_id STRING, 
  product_id STRING,
  quantity INT, 
  unit_price DOUBLE, 
  discount_pct INT, 
  line_total DOUBLE,
  _source_file STRING, 
  _ingested_at TIMESTAMP
) USING DELTA;

# 4. Create the silver quarantine table for order_items in order to send the bad data

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.order_items_quarantine (
  order_item_id STRING, 
  order_id STRING, 
  product_id STRING,
  quantity INT, 
  unit_price DOUBLE, 
  discount_pct INT, 
  line_total DOUBLE,
  _source_file STRING, 
  _ingested_at TIMESTAMP, 
  _quarantine_reason STRING
) USING DELTA;

# 5. Create the silver table with an explicit schema for returns

In [0]:
# ============================================================
# Create the silver.returns table
# ============================================================
# Columns match what the generator produced for returns_*.csv,
# plus our two lineage columns from bronze.
spark.sql("""
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.returns (
  return_id       STRING,
  order_id        STRING,
  order_item_id   STRING,
  return_date     DATE,
  return_reason   STRING,
  refund_amount   DOUBLE,
  _source_file    STRING,
  _ingested_at    TIMESTAMP
) USING DELTA
""")

# 6. Create the silver.customers table with SCD2 columns

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.customers (
  customer_id           STRING,
  first_name            STRING,
  last_name             STRING,
  email                 STRING,
  phone                 STRING,
  address               STRING,
  city                  STRING,
  state                 STRING,
  zip_code              STRING,
  country               STRING,
  segment               STRING,
  row_hash              STRING,      -- fingerprint of tracked attributes, used to detect real changes
  effective_start_date  DATE,        -- when this version became true
  effective_end_date    DATE,        -- when this version stopped being true (NULL = still current)
  is_current            BOOLEAN,     -- fast filter for "give me today's version of each customer"
  _source_file          STRING,
  _ingested_at          TIMESTAMP
) USING DELTA;

In [0]:
# ============================================================
# ⚠️ TODAY-ONLY, STEP 1 — Wipe the mess and start clean
# ============================================================
# Run this ONCE, right now, to undo the accidental "both files
# merged into one batch before silver ever ran" situation.
# You will NEVER run this again after today.


spark.sql("DELETE FROM azuresalesdatabricks.silver.customers")

dbutils.fs.rm(
    "abfss://silver@azuresales.dfs.core.windows.net/_checkpoints/customers/",
    recurse=True
)

In [0]:
# ============================================================
# ⚠️ TODAY-ONLY, STEP 2 — Seed the baseline (400 rows) FIRST
# ============================================================
# We manually call the function ourselves with JUST the original
# customers.csv rows, so silver.customers has a "before" state
# to compare against once the delta file gets processed next.


base_only = (spark.table("azuresalesdatabricks.bronze.customers")
    .filter(~F.col("_source_file").contains("updates"))
)
merge_customers_scd2(base_only, "seed")


# ============================================================
# ⚠️ TODAY-ONLY, STEP 3 — Now apply JUST the delta (15 rows)
# ============================================================
# Because step 2 already seeded their "before" version, this call
# will correctly find matches, close the old rows, and insert
# new versioned rows — real SCD2 history, not a silent overwrite.



delta_only = (spark.table("azuresalesdatabricks.bronze.customers")
    .filter(F.col("_source_file").contains("updates"))
)
merge_customers_scd2(delta_only, "delta")

# 7. products, stores and sales_reps

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.products (
  product_id STRING, product_name STRING, category STRING, subcategory STRING,
  brand STRING, unit_cost DOUBLE, unit_price DOUBLE,
  row_hash STRING, effective_start_date DATE, effective_end_date DATE, is_current BOOLEAN,
  _source_file STRING, _ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.stores (
  store_id STRING, store_name STRING, region STRING, city STRING, state STRING, store_type STRING,
  row_hash STRING, effective_start_date DATE, effective_end_date DATE, is_current BOOLEAN,
  _source_file STRING, _ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS azuresalesdatabricks.silver.sales_reps (
  rep_id STRING, rep_name STRING, email STRING, store_id STRING, region STRING,
  row_hash STRING, effective_start_date DATE, effective_end_date DATE, is_current BOOLEAN,
  _source_file STRING, _ingested_at TIMESTAMP
) USING DELTA;